### OSM STYLED DATASET GENERATION

### Automatic OSM Data Download with Raster Bbox

- This script calls `osm_processor.py` and downloads data according to the given folder.
- For each `image.tif`, OSM data is downloaded and saved in a folder with the same name as the raster file.
- Input Folder:
    ```
    input_folder/
    └── image_001.tif
    └── image_002.tif
    ```
- Output Folder:
    ```
    output_folder/
    ├── image_001/
    │   ├── gis_osm_buildings_a_free_1.shp
    │   └── ...
    ├── image_002/
    │   └── ...
    ```

In [ ]:
!python batch_runner.py --input ".\raster" \
                        --output ".\osm_download" \
                        --ext tif \
                        --recursive


### Automatic OSM Process 
- This script calls `osm_formatter_engine.py` and edits the raw OSM data to convert it into the Geofabrik data structure.
- This code takes the main output folder generated by `osm_processor.py` as the input path. The input main folder structure should be like this:
- Input Folder:
    ```
    input_folder/
    ├── image_001/
    │   ├── gis_osm_buildings_a_free_1.shp
    │   └── ...
    ├── image_002/
    │   └── ...
    ```
- Output Folder:
    ```
    output_folder/
    ├── image_001/
    │   ├── gis_osm_buildings_a_free_1.shp
    │   └── ...
    ├── image_002/
    │   └── ...
    ```

In [ ]:
!python batch_formatter.py -i ".\osm_download" \
                           -o ".\osm_download_formatted"

### Merged Formatted Shapefiles

- This command finds shapefiles with the same layer name across subfolders.
- It merges them into a single shapefile for each OSM category.
- This step is useful when the folders represent tiles of the same study area.
- For independent images or different study areas, this step can be skipped to keep the outputs separated by image.

In [ ]:
!python merge_shp.py --base_dir ".\osm_download_formatted" \
                    --output_dir ".\osm_download_formatted_merged"


### Apply OSM Styles and Split Tiles

This notebook cell runs the run.bat file from Python and executes the OSM tile generation workflow through the QGIS Python environment.

In this notebook:

- Path to run.bat
- --shp_dir
- --qml_dir
- --raster_dir
- --output

In run.bat:
QGIS_DIR: main QGIS installation directory
MY_SCRIPT: full path of osm_style_processor.py

Note: QGIS_DIR is defined inside run.bat. It is not defined in this notebook cell, but the notebook depends on run.bat being correctly configured.

In [ ]:
import subprocess

cmd = [
    "cmd.exe", "/c",
    r".\run.bat",
    "--shp_dir",    r".\osm_download_formatted_merged",
    "--qml_dir",    r".\Styles",
    "--raster_dir", r".\raster",
    "--output",     r".\tiles",
    "--tile_size",  "256",
    "--dpi",        "96",
]

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    print(line, end="")
process.wait()
print("Exit code:", process.returncode)

### Cleanup tiles

This step removes raster–OSM tile pairs that contain black/no-data areas, especially near image boundaries or where OSM coverage is incomplete.

Tiles with 1% or more black pixels are moved to the deleted/ folder.

In [ ]:
!python cleanup_tiles.py --base_dir ".\tiles" --threshold 0.01